In [1]:
# OPTIONAL: Install extras for tools/agents
!pip install -qU langchain langchain-community langchain-openai duckduckgo-search wikipedia numexpr python-dotenv


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Tutorial: Tools, Chains, Agents — Core Concepts (CRM Mini-App)

We’ll implement a tiny in-memory CRM to ground the concepts:
- **Tool**: `add_contact` and `get_contact` that mutate/read a store.
- **Chain**: LLM formatting/summarization of tool results.
- **Agent**: LLM decides whether to add or read and routes accordingly.

Goal: show how tools (actions), chains (deterministic formatting), and agents (routing) fit together in a practical example.


### Step 1: Imports and model
Small, deterministic config; keep creds outside the notebook.


In [4]:
from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# --------------------------------------------------
# OpenRouter configuration
# --------------------------------------------------

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key (hidden): ")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# --------------------------------------------------
# LLM
# --------------------------------------------------

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    seed=42,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL
)

# --------------------------------------------------
# Prompt
# --------------------------------------------------

prompt = PromptTemplate.from_template(
    "Explain the following concept in simple terms:\n\n{topic}"
)

# --------------------------------------------------
# Create chain using LCEL
# --------------------------------------------------

chain = prompt | llm

# --------------------------------------------------
# Run the chain
# --------------------------------------------------

response = chain.invoke({
    "topic": "Artificial Intelligence"
})

print(response.content)

Enter your OpenRouter API key (hidden): ··········
Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines or software that can perform tasks that usually require human intelligence. This includes things like understanding language, recognizing images, solving problems, and making decisions. 

In simple terms, you can think of AI as teaching computers to think and learn like humans do, so they can help us with various tasks, from answering questions to driving cars.


### Step 2: CRM Tools (in-memory)
We’ll create two tools over a simple in-memory store:
- `add_contact(name, email, phone)` → adds or updates a contact
- `get_contact(name)` → returns contact details or a not-found message


In [5]:
from typing import Dict, Optional

crm_store: Dict[str, Dict[str, str]] = {}

def add_contact(name: str, email: Optional[str] = None, phone: Optional[str] = None) -> Dict[str, str]:
    """Add or update a contact in the in-memory store and return the record."""
    if not name or not isinstance(name, str):
        return {"status": "error", "message": "Name is required and must be a string."}
    record = crm_store.get(name.strip(), {})
    if email:
        record["email"] = email.strip()
    if phone:
        record["phone"] = phone.strip()
    record["name"] = name.strip()
    crm_store[name.strip()] = record
    return {"status": "ok", "message": "Contact saved.", "contact": record}

def get_contact(name: str) -> Dict[str, str]:
    """Retrieve a contact by name from the in-memory store."""
    if not name or not isinstance(name, str):
        return {"status": "error", "message": "Name is required and must be a string."}
    record = crm_store.get(name.strip())
    if record:
        return {"status": "ok", "contact": record}
    return {"status": "not_found", "message": f"No contact found for '{name.strip()}'"}

# Quick sanity check
print(add_contact("Alice", email="alice@example.com"))
print(get_contact("Alice"))


{'status': 'ok', 'message': 'Contact saved.', 'contact': {'email': 'alice@example.com', 'name': 'Alice'}}
{'status': 'ok', 'contact': {'email': 'alice@example.com', 'name': 'Alice'}}


### Step 3: LLM Formatting Chain
We’ll use a deterministic chain to format contact data (or errors) from the tools into a concise response.


In [7]:
from langchain_core.prompts import PromptTemplate

format_prompt = PromptTemplate.from_template(
    """
    You are a helpful CRM assistant. Given the raw JSON-like dict below,
    produce a concise, user-facing one-liner.

    If status is ok and contact exists,
    return "Name — email, phone" (omit missing fields).

    If not_found or error,
    return the message as-is.

    Input:
    {tool_result}
    """
)

# New LangChain syntax
format_chain = format_prompt | llm


def format_tool_result(tool_result: dict) -> str:
    response = format_chain.invoke({
        "tool_result": str(tool_result)
    })
    return response.content


# Sanity check formatting
print(format_tool_result(get_contact("Alice")))

Alice — alice@example.com


### Step 4: Agent Router (choice: add vs read)
An agent chooses which tool to call based on the user's intent:
- If the user asks to save/add/update a contact → call `add_contact`
- If the user asks to look up/find/show a contact → call `get_contact`
We'll use a simple intent classifier and parameter extractor.


In [9]:
from typing import Literal, Tuple, Dict
import re

from langchain_core.prompts import PromptTemplate

Intent = Literal["add", "read", "unknown"]


# --------------------------------------------------
# Intent detection
# --------------------------------------------------

intent_prompt = PromptTemplate.from_template(
    """
    Determine if the user wants to add/update a contact or read a contact.

    Reply with only one word: add, read, or unknown.

    User: {text}
    """
)

intent_chain = intent_prompt | llm


# --------------------------------------------------
# Extract fields for adding a contact
# --------------------------------------------------

extract_add_prompt = PromptTemplate.from_template(
    """
    Extract fields for adding a contact from the text.

    Use JSON with keys:
    name, email, phone

    Missing fields should be null.

    Text: {text}
    """
)

extract_add_chain = extract_add_prompt | llm


# --------------------------------------------------
# Extract contact name for reading
# --------------------------------------------------

extract_read_prompt = PromptTemplate.from_template(
    """
    Extract the contact name to look up.

    Reply with just the name string.

    Text: {text}
    """
)

extract_read_chain = extract_read_prompt | llm

In [12]:
from typing import Literal, Tuple, Dict
import re
import json

Intent = Literal["add", "read", "unknown"]


def classify_intent(text: str) -> Intent:
    response = intent_chain.invoke({
        "text": text
    })

    out = response.content.strip().lower()

    if out.startswith("add"):
        return "add"

    if (
        out.startswith("read")
        or out.startswith("lookup")
        or out.startswith("get")
    ):
        return "read"

    return "unknown"


def parse_add_fields(text: str) -> Tuple[str, Dict[str, str]]:
    response = extract_add_chain.invoke({
        "text": text
    })

    raw = response.content.strip()

    # Try to extract JSON from the response
    name = None
    email = None
    phone = None

    try:
        # Remove markdown code fences if present
        cleaned = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            raw,
            flags=re.IGNORECASE
        ).strip()

        # Find JSON object
        match = re.search(r"\{.*\}", cleaned, re.S)

        if match:
            js = json.loads(match.group(0))

            name = js.get("name")
            email = js.get("email")
            phone = js.get("phone")

    except Exception:
        # Fallback to regex parsing
        m_name = re.search(
            r"name[:=]\s*([\w\s]+)",
            raw,
            re.I
        )

        if m_name:
            name = m_name.group(1).strip()

        m_email = re.search(
            r"[\w\.-]+@[\w\.-]+",
            raw
        )

        if m_email:
            email = m_email.group(0)

        m_phone = re.search(
            r"(\+?\d[\d\s\-]{6,}\d)",
            raw
        )

        if m_phone:
            phone = m_phone.group(1).strip()

    return name, {
        "email": email,
        "phone": phone
    }


def parse_read_name(text: str) -> str:
    response = extract_read_chain.invoke({
        "text": text
    })

    name = response.content.strip()

    # Remove surrounding quotes
    name = re.sub(
        r'^[\"\']|[\"\']$',
        "",
        name
    )

    return name.strip()


def crm_agent(user_input: str) -> str:

    intent = classify_intent(user_input)

    if intent == "add":

        name, fields = parse_add_fields(user_input)

        if not name:
            return "Please provide a contact name to add/update."

        result = add_contact(
            name=name,
            email=fields.get("email"),
            phone=fields.get("phone")
        )

        return format_tool_result(result)

    if intent == "read":

        name = parse_read_name(user_input)

        if not name:
            return "Please provide the contact name to look up."

        result = get_contact(name)

        return format_tool_result(result)

    return (
        "I can add or look up contacts. "
        "Try: 'Add John Doe john@x.com +1 555 123 4567' "
        "or 'Find John Doe'."
    )

In [13]:
# Demos
print(crm_agent("Add Alice Cooper with email alice@co.com and phone +1 202 555 0142"))
print(crm_agent("Find Alice Cooper"))

Alice Cooper — alice@co.com, +1 202 555 0142
Alice Cooper — alice@co.com, +1 202 555 0142


### Step 5: When to use which?
- **Chain**: deterministic workflow, known steps. Faster, cheaper, easier to test.
- **Agent**: dynamic tasks, unknown number/order of steps, tool choice.
- Start with chains, graduate to agents when branching logic explodes.

You now have intuition + a tiny agent sketch. In later sections, we’ll wire full LangChain Tools and Agents.
